In [14]:
import sqlite3
import pandas as pd
import os
import numpy as np

In [15]:
conn = sqlite3.connect("customer_shopping_behavior.db")
cursor = conn.cursor()

In [16]:
pd.read_sql_query("" \
"SELECT name " \
"FROM sqlite_master " \
"WHERE type='table';", conn)

,name
0,cleaned_customer_shopping_behavior
1,customer_shopping_behavior


In [17]:
pd.read_sql_query("" \
"SELECT * FROM " \
"cleaned_customer_shopping_behavior " \
"LIMIT 5;", conn)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,middle_aged,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,young_adult,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,middle_aged,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,young_adult,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,middle_aged,365.0


In [19]:
# 1.  what is the total revenue generated by gender?
pd.read_sql_query("""SELECT gender, SUM(purchase_amount) AS revenue
FROM cleaned_customer_shopping_behavior GROUP BY gender""", conn)

,gender,revenue
0,Female,75191
1,Male,157890


In [20]:
# 2. which age group has the highest average purchase amount?
pd.read_sql_query("""SELECT age_group, AVG(purchase_amount) AS avg_purchase
FROM cleaned_customer_shopping_behavior GROUP BY age_group ORDER BY avg_purchase DESC LIMIT 1;""", conn)

,age_group,avg_purchase
0,young_adult,60.450389


In [33]:
# 3. Which customer used a discount but still spent more than the average purchase amount?
pd.read_sql_query(""" 
SELECT customer_id, purchase_amount from cleaned_customer_shopping_behavior
                  WHERE discount_applied = 'Yes' ANd purchase_amount > (SELECT AVG(purchase_amount) FROM cleaned_customer_shopping_behavior);""", conn)

,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


In [24]:
# 4. what ar ethe top 5 categories with the highest total revenue?
pd.read_sql_query("""SELECT category, SUM(purchase_amount) AS total_revenue
FROM cleaned_customer_shopping_behavior GROUP BY category ORDER BY total_revenue DESC LIMIT 5;""", conn)

,category,total_revenue
0,Clothing,104264
1,Accessories,74200
2,Footwear,36093
3,Outerwear,18524


In [39]:
# 5. what are the top 5 categories with the highest average rating ?
pd.read_sql_query("""
SELECT 
item_purchased AS category, 
ROUND(AVG(review_rating), 2) AS avg_rating 
FROM cleaned_customer_shopping_behavior 
GROUP BY item_purchased 
ORDER BY avg_rating DESC LIMIT 5;""", conn)

,category,avg_rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,T-shirt,3.78


In [30]:
# 6. compare the average purchase amount between standard and express shipping methods
pd.read_sql_query("""SELECT shipping_type, AVG(purchase_amount) AS avg_purchase
FROM cleaned_customer_shopping_behavior 
                WHERE shipping_type IN ('Standard', 'Express')
                GROUP BY shipping_type;""", conn)

,shipping_type,avg_purchase
0,Express,60.475232
1,Standard,58.460245


In [42]:
# 7 Do subscribers Customers spend more Compare average spend and total revenues Between subscribers and the non subscribers
pd.read_sql_query("""
SELECT 
subscription_status,
COUNT(*) AS total_customers,
ROUND(AVG(purchase_amount), 2) AS avg_purchase,
SUM(purchase_amount) AS total_revenue
FROM cleaned_customer_shopping_behavior
GROUP BY subscription_status;""", conn)

,subscription_status,total_customers,avg_purchase,total_revenue
0,No,2847,59.87,170436
1,Yes,1053,59.49,62645


In [41]:
# 8 Which five products have the highest percentage of purchases With discount applied
pd.read_sql_query("""
SELECT
item_purchased AS product,
ROUND((SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS discount_percentage
FROM cleaned_customer_shopping_behavior 
GROUP BY item_purchased
ORDER BY discount_percentage DESC LIMIT 5;""", conn)

,product,discount_percentage
0,Hat,50.00
1,Sneakers,49.66
2,Coat,49.07
3,Sweater,48.17
4,Pants,47.37


In [43]:
# 9 Segment the customers into new returning and loyal based on their total number of previous purchases and show the count of each segment
pd.read_sql_query("""
with customer_segments AS (
SELECT
customer_id,
previous_purchases,
CASE
    WHEN previous_purchases = 1 THEN 'New'
    WHEN previous_purchases BETWEEN 2 AND 5 THEN 'returning'
    ELSE 'loyal'
END AS segment
FROM cleaned_customer_shopping_behavior
)
                  
SELECT 
segment, 
COUNT(*) AS count
FROM customer_segments
GROUP BY segment;""", conn)

,segment,count
0,New,83
1,loyal,3476
2,returning,341


In [ ]:
# 10 What are the top three most purchased products within each category 
# pd.read_sql_query("""
# with item_count as (
# SELECT 
# item_purchased,
# category,
# COUNT(*) AS purchase_count
# FROM cleaned_customer_shopping_behavior
# GROUP BY item_purchased, category
# ),
#
# """this the main logic"""     
# ranked_items as ( 
# SELECT
# item_purchased,
# category,
# purchase_count,
# RANK() OVER (PARTITION BY category ORDER BY purchase_count DESC) AS rank
# FROM item_count
# )
#
# SELECT
# item_purchased,
# category,
# purchase_count
# FROM ranked_items
# WHERE rank <= 3
# ORDER BY category, purchase_count DESC;""", conn)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (2041546465.py, line 2)